# HRM LoRA Fine-Tuning: 9×9 Sudoku Extreme

This notebook fine-tunes the pre-trained HRM (Hierarchical Reasoning Model) on 9×9 Sudoku Extreme puzzles using LoRA (Low-Rank Adaptation).

**Key Settings (optimized for Kaggle T4 GPU, 12h limit):**
- Dataset: 500 puzzles × 100 augmentations (reduced from 1000×1000 to avoid timeout)
- LoRA rank: 8, alpha: 16 (~558K trainable params out of 27M)
- Epochs: 4, eval every 2 epochs
- Batch size: 8

**Estimated Runtime: ~3-4 hours**

## 1. Setup Environment

In [ ]:
!git clone https://github.com/jagan25-mj/NHRT.git
%cd NHRT
!pip install setuptools wheel
!pip install einops tqdm pydantic wandb omegaconf hydra-core huggingface_hub peft argdantic coolname --quiet


## 2. Build 9×9 Sudoku Extreme Dataset (Reduced Size)

Using `--subsample-size 500 --num-aug 100` instead of 1000/1000 to save ~2 hours on dataset building.

In [ ]:
!python dataset/build_sudoku_dataset.py --output-dir data/sudoku-extreme-500-aug-100 --subsample-size 500 --num-aug 100

## 3. Download Pre-trained HRM Checkpoint

In [ ]:
import os
from huggingface_hub import snapshot_download

os.makedirs('checkpoints/HRM-checkpoint-sudoku-extreme', exist_ok=True)
snapshot_download(
    repo_id='sapientinc/HRM-checkpoint-sudoku-extreme',
    local_dir='checkpoints/HRM-checkpoint-sudoku-extreme'
)
print("Checkpoint downloaded!")
print("Files:", os.listdir('checkpoints/HRM-checkpoint-sudoku-extreme'))

## 4. Write the LoRA Fine-Tuning Script

This creates `finetune_lora.py` which:
- Loads the pre-trained checkpoint
- Freezes all base weights
- Injects LoRA adapters into attention and MLP layers
- Trains only the LoRA weights + Q-head (~558K params)

In [ ]:
%%writefile finetune_lora.py
from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil

import torch
import torch.distributed as dist
from torch import nn
from torch.utils.data import DataLoader

import tqdm
import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from torch.optim import AdamW

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed
from models.layers import CastedLinear
import torch.nn.functional as F

original_casted_linear_forward = CastedLinear.forward

def lora_forward(self, input: torch.Tensor) -> torch.Tensor:
    base_out = original_casted_linear_forward(self, input)
    if hasattr(self, 'lora_A') and hasattr(self, 'lora_B'):
        lora_A_casted = self.lora_A.to(input.dtype)
        lora_B_casted = self.lora_B.to(input.dtype)
        lora_out = F.linear(F.linear(input, lora_A_casted), lora_B_casted) * self.scaling
        return base_out + lora_out.to(base_out.dtype)
    return base_out

CastedLinear.forward = lora_forward

def apply_lora_and_freeze(model, r=8, alpha=16):
    for param in model.parameters():
        param.requires_grad = False
    
    for name, module in model.named_modules():
        if isinstance(module, CastedLinear):
            if "qkv_proj" in name or "o_proj" in name or "gate_up_proj" in name or "down_proj" in name:
                module.lora_A = nn.Parameter(torch.zeros((r, module.weight.shape[1]), device=module.weight.device, dtype=module.weight.dtype))
                module.lora_B = nn.Parameter(torch.zeros((module.weight.shape[0], r), device=module.weight.device, dtype=module.weight.dtype))
                nn.init.kaiming_uniform_(module.lora_A, a=math.sqrt(5))
                nn.init.zeros_(module.lora_B)
                module.scaling = alpha / r
                module.lora_A.requires_grad = True
                module.lora_B.requires_grad = True
                
    for name, param in model.named_parameters():
        if "q_head" in name:
            param.requires_grad = True


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    name: str
    loss: LossConfig


class PretrainConfig(pydantic.BaseModel):
    arch: ArchConfig
    load_checkpoint: Optional[str] = None
    data_path: str
    global_batch_size: int
    epochs: int
    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int
    weight_decay: float
    beta1: float
    beta2: float
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    checkpoint_path: Optional[str] = None
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    eval_save_outputs: List[str] = []


@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any
    step: int
    total_steps: int


def create_dataloader(config, split, rank, world_size, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed, dataset_path=config.data_path,
        rank=rank, num_replicas=world_size, **kwargs
    ), split=split)
    dataloader = DataLoader(dataset, batch_size=None, num_workers=1,
                           prefetch_factor=8, pin_memory=True, persistent_workers=True)
    return dataloader, dataset.metadata


def create_model(config, train_metadata, world_size):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,
        batch_size=config.global_batch_size // world_size,
        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=max(train_metadata.num_puzzle_identifiers, train_metadata.blank_identifier_id + 1),
        causal=False
    )
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)
    with torch.device("cpu"):
        model = model_cls(model_cfg)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model, dynamic=False)
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    if config.load_checkpoint is not None:
        print(f"Loading checkpoint from {config.load_checkpoint}...")
        state_dict = torch.load(config.load_checkpoint, map_location="cpu")
        clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(clean_state_dict, strict=False)
        print("Checkpoint loaded.")

    print("Applying LoRA...")
    apply_lora_and_freeze(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"LoRA applied. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    optimizers = [
        CastedSparseEmbeddingSignSGD_Distributed(
            model.model.puzzle_emb.buffers(), lr=0,
            weight_decay=config.puzzle_emb_weight_decay, world_size=world_size),
        AdamW(model.parameters(), lr=0, weight_decay=config.weight_decay,
              betas=(config.beta1, config.beta2))
    ]
    return model, optimizers, [config.puzzle_emb_lr, config.lr]


def cosine_schedule_with_warmup_lr_lambda(current_step, *, base_lr, num_warmup_steps, num_training_steps, min_ratio=0.0, num_cycles=0.5):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config, train_metadata, world_size):
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)
    return TrainState(step=0, total_steps=total_steps, model=model, optimizers=optimizers, optimizer_lrs=optimizer_lrs, carry=None)


def save_train_state(config, train_state):
    if config.checkpoint_path is None: return
    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))
    print(f"Checkpoint saved at step {train_state.step}")


def compute_lr(base_lr, config, train_state):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step, base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps, min_ratio=config.lr_min_ratio)


def train_batch(config, train_state, batch, global_batch_size, rank, world_size):
    train_state.step += 1
    if train_state.step > train_state.total_steps: return
    batch = {k: v.cuda() for k, v in batch.items()}
    if train_state.carry is None:
        with torch.device("cuda"):
            train_state.carry = train_state.model.initial_carry(batch)
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])
    ((1 / global_batch_size) * loss).backward()
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None: dist.all_reduce(param.grad)
    lr_this_step = None
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)
        for pg in optim.param_groups: pg['lr'] = lr_this_step
        optim.step(); optim.zero_grad()
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())
        metric_keys = list(sorted(metrics.keys()))
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1: dist.reduce(metric_values, dst=0)
        if rank == 0:
            mv = metric_values.cpu().numpy()
            rm = {k: mv[i] for i, k in enumerate(metric_keys)}
            count = max(rm["count"], 1)
            rm = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in rm.items()}
            rm["train/lr"] = lr_this_step
            return rm


def evaluate(config, train_state, eval_loader, eval_metadata, rank, world_size):
    with torch.no_grad():
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        all_preds = {}
        metric_keys = []
        metric_values = None
        carry = None
        for set_name, batch, global_batch_size in eval_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)
            while True:
                carry, _, metrics, preds, all_finish = train_state.model(carry=carry, batch=batch, return_keys=config.eval_save_outputs)
                if all_finish: break
            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        all_preds.setdefault(k, []).append(v.cpu())
            del carry, preds, batch, all_finish
            set_id = set_ids[set_name]
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))
                metric_values = torch.zeros((len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda")
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
        if len(all_preds) and config.checkpoint_path is not None:
            all_preds = {k: torch.cat(v, dim=0) for k, v in all_preds.items()}
            os.makedirs(config.checkpoint_path, exist_ok=True)
            torch.save(all_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}"))
        if metric_values is not None:
            if world_size > 1: dist.reduce(metric_values, dst=0)
            if rank == 0:
                rm = metric_values.cpu().numpy()
                rm = {sn: {mn: rm[si, mi] for mi, mn in enumerate(metric_keys)} for si, sn in enumerate(set_ids)}
                for sn, m in rm.items():
                    count = m.pop("count")
                    rm[sn] = {k: v / count for k, v in m.items()}
                return rm


def save_code_and_config(config):
    if config.checkpoint_path is None or wandb.run is None: return
    os.makedirs(config.checkpoint_path, exist_ok=True)
    for cf in [get_model_source_path(config.arch.name), get_model_source_path(config.arch.loss.name)]:
        if cf: shutil.copy(cf, os.path.join(config.checkpoint_path, os.path.basename(cf)))
    with open(os.path.join(config.checkpoint_path, "all_config.yaml"), "wt") as f:
        yaml.dump(config.model_dump(), f)
    wandb.run.log_code(config.checkpoint_path)


def load_synced_config(hydra_config, rank, world_size):
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_path).capitalize()} ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)
        objects = [config]
    if world_size > 1: dist.broadcast_object_list(objects, src=0)
    return objects[0]


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config):
    RANK = 0; WORLD_SIZE = 1
    if "LOCAL_RANK" in os.environ:
        dist.init_process_group(backend="nccl")
        RANK = dist.get_rank(); WORLD_SIZE = dist.get_world_size()
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)
    torch.random.manual_seed(config.seed + RANK)
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter
    assert config.epochs % train_epochs_per_iter == 0
    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader, eval_metadata = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)
    progress_bar = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)
        wandb.init(project=config.project_name, name=config.run_name, config=config.model_dump(), settings=wandb.Settings(_disable_stats=True))
        wandb.log({"num_params": sum(x.numel() for x in train_state.model.parameters())}, step=0)
        save_code_and_config(config)
    for _iter_id in range(total_iters):
        print(f"\n{'='*60}")
        print(f"TRAINING EPOCH {_iter_id * train_epochs_per_iter + 1} to {(_iter_id + 1) * train_epochs_per_iter}")
        print(f"{'='*60}")
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)
            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)
        if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
            save_train_state(config, train_state)
        try:
            train_state.model.eval()
            metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)
            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                print(f"\n{'='*60}")
                print(f"EVALUATION RESULTS at step {train_state.step}")
                print(f"{'='*60}")
                for set_name, set_metrics in metrics.items():
                    print(f"  Set: {set_name}")
                    for k, v in set_metrics.items():
                        print(f"    {k}: {v:.4f}")
                print(f"{'='*60}\n")
        except Exception as e:
            print(f"WARNING: Evaluation failed at step {train_state.step}: {e}")
            print("Checkpoint was already saved. Training will continue.")
    print("\n" + "#"*60)
    print("TRAINING COMPLETE!")
    print("#"*60)
    if dist.is_initialized(): dist.destroy_process_group()
    wandb.finish()

if __name__ == "__main__":
    launch()

## 5. Run LoRA Fine-Tuning

Fine-tuning the pre-trained 9×9 Sudoku model with LoRA adapters.

- **epochs=4**: 4 full passes through the dataset
- **eval_interval=2**: Evaluate every 2 epochs
- **global_batch_size=8**: Fits in T4 GPU VRAM
- **lr=1e-5**: Conservative learning rate for fine-tuning

In [ ]:
!WANDB_MODE=disabled python finetune_lora.py \
    data_path=data/sudoku-extreme-500-aug-100 \
    epochs=4 \
    eval_interval=2 \
    lr=1e-5 \
    puzzle_emb_lr=1e-5 \
    weight_decay=1.0 \
    puzzle_emb_weight_decay=1.0 \
    global_batch_size=8 \
    checkpoint_every_eval=True \
    +load_checkpoint=checkpoints/HRM-checkpoint-sudoku-extreme/checkpoint

## 6. Final Summary

In [ ]:
import os

print("=" * 60)
print("9x9 SUDOKU LoRA FINE-TUNING COMPLETE")
print("=" * 60)

print("\nSaved checkpoints:")
for root, dirs, files in os.walk("checkpoints"):
    for f in files:
        if f.startswith("step_"):
            full_path = os.path.join(root, f)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"  {full_path} ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("SCROLL UP to find 'EVALUATION RESULTS' sections")
print("Key metrics to note:")
print("  - exact_accuracy: % of puzzles solved perfectly")
print("  - accuracy: per-cell accuracy")
print("=" * 60)